# 第8章 最適化と実装・運用 ― サンプルコード
『AI実装検定 A級・S級 完全版』の第8章のリストを、本文と同じ順に実行できます。

**使い方**：上のメニュー［ランタイム］→［すべてのセルを実行］。CPU のランタイムで動きます（GPU は不要）。

**本書の動作確認環境**（Python 3.13.17）：NumPy 2.5.3、pandas 3.0.5、scikit-learn 1.9.1、Matplotlib 3.11.2、Seaborn 0.13.2、PyTorch 2.14.1、TensorFlow 2.21.0、Keras 3.15.1、OpenCV 5.0.0.93、scikit-image 0.26.0、NLTK 3.10.4、Transformers 5.20.2、torchvision 0.29.1

In [ ]:
# 版をそろえたいとき（任意）：本書と同じ版を入れます。通常は Colab の標準版でそのまま動きます。
# !pip install -q numpy==2.5.3 pandas==3.0.5 scikit-learn==1.9.1 matplotlib==3.11.2 seaborn==0.13.2 torch==2.14.1 tensorflow-cpu==2.21.0 keras==3.15.1 opencv-python==5.0.0.93 scikit-image==0.26.0 nltk==3.10.4 transformers==5.20.2 torchvision==0.29.1
import sys; print(sys.version)


## リスト8.1　70 億パラメータのモデルの、重みだけのメモリ

In [ ]:
for name, bytes_per in [("FP32", 4), ("FP16", 2), ("INT8", 1), ("INT4", 0.5)]:
    params = 7e9                                       # 70 億パラメータのモデル
    print(name, params * bytes_per / 1e9, "GB")        # 重みだけのメモリ（活性化・KV キャッシュは別）

**本書に載っている出力**
```
FP32 28.0 GB
FP16 14.0 GB
INT8 7.0 GB
INT4 3.5 GB
```

## リスト8.2　手で書いた INT8 量子化（5 つの重み）

In [ ]:
import numpy as np

w = np.array([-1.0, -0.2, 0.0, 0.5, 2.0], dtype=np.float32)
qmin, qmax = -128, 127
scale = (w.max() - w.min()) / (qmax - qmin)            # 刻み幅
zp = int(round(qmin - w.min() / scale))                # ゼロ点：実数 0 に対応する整数
q = np.clip(np.round(w / scale) + zp, qmin, qmax).astype(np.int8)
back = (q.astype(np.float32) - zp) * scale             # 元に戻す（逆量子化）
print(round(float(scale), 5), zp)
print(q.tolist())
print([round(float(v), 3) for v in back])
print(round(float(np.abs(back - w).max()), 5), scale / 2 >= np.abs(back - w).max())   # 誤差は刻み幅の半分以内

**本書に載っている出力**
```
0.01176 -43
[-128, -60, -43, -1, 127]
[-1.0, -0.2, 0.0, 0.494, 2.0]
0.00588 True
```

## リスト8.3　動的量子化によるサイズの変化

In [ ]:
import io
import warnings
import torch
from torch import nn

warnings.filterwarnings("ignore")
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10)).eval()
qmodel = torch.ao.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)

def size(m):
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return buf.getbuffer().nbytes

s1, s2 = size(model), size(qmodel)
print(s1, s2, round(s1 / s2, 2))
x = torch.randn(4, 256)
print(round((model(x) - qmodel(x)).abs().max().item(), 4) < 0.1)   # 出力の差は小さい
print(type(qmodel[0]).__module__)

**本書に載っている出力**
```
275677 72717 3.79
True
torch.ao.nn.quantized.dynamic.modules.linear
```

## リスト8.4　L1 ノルムによる枝刈り（50%）

In [ ]:
import torch
from torch import nn
from torch.nn.utils import prune

torch.manual_seed(0)
lin = nn.Linear(100, 50)
prune.l1_unstructured(lin, name="weight", amount=0.5)  # 絶対値の小さい 50% を 0 にする
w = lin.weight
print(round((w == 0).float().mean().item(), 3))
print(w.shape, lin.weight_mask.shape)                   # 形は変わらない（マスクで 0 にしているだけ）
print(sorted(n for n, _ in lin.named_parameters()))     # 元の重みは weight_orig に残る
prune.remove(lin, "weight")                             # マスクを焼き込んで恒久化
print(sorted(n for n, _ in lin.named_parameters()))

**本書に載っている出力**
```
0.5
torch.Size([50, 100]) torch.Size([50, 100])
['bias', 'weight_orig']
['bias', 'weight']
```

## リスト8.5　温度 T と蒸留損失の大きさ

In [ ]:
import numpy as np

def softmax(z, T=1.0):
    e = np.exp(z / T - (z / T).max())
    return e / e.sum()

teacher = np.array([4.0, 1.0, 0.0])
student = np.array([2.0, 1.0, 0.5])
for T in (1.0, 4.0):
    pt, ps = softmax(teacher, T), softmax(student, T)
    kl = float((pt * np.log(pt / ps)).sum())
    print(T, pt.round(3).tolist(), round(kl, 4), round(kl * T * T, 4))   # T² を掛けて勾配の大きさをそろえる

**本書に載っている出力**
```
1.0 [0.936, 0.047, 0.017] 0.2624 0.2624
4.0 [0.543, 0.257, 0.2] 0.0394 0.6312
```

## リスト8.6　デバイスの選択と、モデル・データの配置

In [ ]:
import torch
from torch import nn

print(torch.cuda.is_available())                        # この実行環境には GPU がない
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = nn.Linear(4, 2).to(device)                      # モデルとデータは同じデバイスに置く
x = torch.randn(3, 4).to(device)
print(model(x).device, x.device)

**本書に載っている出力**
```
False
cpu cpu
```

## リスト8.7　trace と script の違い（データ依存の分岐）

In [ ]:
import torch
from torch import nn

class Gate(nn.Module):
    def forward(self, x):
        if x.sum() > 0:                                 # データに依存する分岐
            return x * 2
        return x - 1

m = Gate()
traced = torch.jit.trace(m, torch.ones(3), check_trace=False)   # 実行された経路だけを記録する
scripted = torch.jit.script(m)                                   # コードを解析して分岐も保存する
neg = -3 * torch.ones(3)
print(m(neg).tolist(), traced(neg).tolist(), scripted(neg).tolist())

**本書に載っている出力**
```
[-4.0, -4.0, -4.0] [-6.0, -6.0, -6.0] [-4.0, -4.0, -4.0]
```

## リスト8.8　ONNX への変換と ONNX Runtime での推論

In [ ]:
import os, tempfile, warnings
import numpy as np
import torch
from torch import nn
import onnxruntime as ort

warnings.filterwarnings("ignore")
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3)).eval()
x = torch.randn(5, 4)
path = os.path.join(tempfile.mkdtemp(), "m.onnx")
torch.onnx.export(model, (x,), path, input_names=["x"], output_names=["y"],
                  dynamic_axes={"x": {0: "batch"}}, dynamo=False)     # バッチの大きさを可変にする
sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
y_ort = sess.run(None, {"x": x.numpy()})[0]
print(np.allclose(y_ort, model(x).detach().numpy(), atol=1e-5))
print(sess.run(None, {"x": np.zeros((2, 4), dtype=np.float32)})[0].shape)   # 別のバッチサイズでも動く
print([i.name for i in sess.get_inputs()], [o.name for o in sess.get_outputs()])

**本書に載っている出力**
```
True
(2, 3)
['x'] ['y']
```

## リスト8.9　FastAPI の API をテストクライアントで確認

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    features: list[float]

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(item: Item):
    score = sum(item.features) / len(item.features)     # 本来はここでモデルを呼ぶ
    return {"label": int(score > 0.5), "score": round(score, 3)}

client = TestClient(app)                                # サーバを起動せずにテストできる
print(client.get("/health").status_code, client.get("/health").json())
r = client.post("/predict", json={"features": [0.2, 0.9, 0.7]})
print(r.status_code, r.json())
print(client.post("/predict", json={"features": "abc"}).status_code)   # 型が違う入力は 422

**本書に載っている出力**
```
200 {'status': 'ok'}
200 {'label': 1, 'score': 0.6}
422
```

## リスト8.10　Dockerfile の例（モデルを提供するコンテナ）

（本書では未実行のリストです。環境によっては追加のインストールや通信が必要です。）

```
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt   # 依存関係を固定して入れる
COPY model.onnx app.py ./
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

（Python ではないので、実行せず表示だけにしています。）

## リスト8.11　イメージのビルドと、コンテナの起動

（本書では未実行のリストです。環境によっては追加のインストールや通信が必要です。）

```
docker build -t my-model:1.0 .
docker run -p 8000:8000 my-model:1.0
```

（Python ではないので、実行せず表示だけにしています。）

## リスト8.12　PSI による分布のずれの検出

In [ ]:
import numpy as np

def psi(expected, actual, bins=10):
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected)
    a = np.histogram(actual, edges)[0] / len(actual)
    a = np.clip(a, 1e-6, None)
    return float(((a - e) * np.log(a / e)).sum())

rng = np.random.default_rng(0)
train = rng.normal(0, 1, 5000)
same = rng.normal(0, 1, 5000)
shifted = rng.normal(0.5, 1, 5000)                      # 平均がずれた本番データ
print(round(psi(train, same), 3), round(psi(train, shifted), 3))

**本書に載っている出力**
```
0.005 0.254
```

## リスト8.13　学習率の推移（StepLR・Cosine・ウォームアップ）

In [ ]:
import torch
from torch import nn, optim
from torch.optim import lr_scheduler as ls

def run(make_sched, epochs=6):
    opt = optim.SGD(nn.Linear(1, 1).parameters(), lr=0.1)
    sch = make_sched(opt)
    lrs = []
    for _ in range(epochs):
        lrs.append(round(opt.param_groups[0]["lr"], 4))
        opt.step()
        sch.step()
    return lrs

print(run(lambda o: ls.StepLR(o, step_size=2, gamma=0.1)))
print(run(lambda o: ls.CosineAnnealingLR(o, T_max=4)))
# ウォームアップ：最初の 3 エポックで 0 から lr まで上げる
print(run(lambda o: ls.LambdaLR(o, lambda e: min(1.0, (e + 1) / 3))))

**本書に載っている出力**
```
[0.1, 0.1, 0.01, 0.01, 0.001, 0.001]
[0.1, 0.0854, 0.05, 0.0146, 0.0, 0.0146]
[0.0333, 0.0667, 0.1, 0.1, 0.1, 0.1]
```

## リスト8.14　ResNet-18 の最後の層を作り直し、学習するパラメータ数を数える

In [ ]:
import torch
from torch import nn
from torchvision.models import resnet18

net = resnet18(weights=None)                # 実際は weights="DEFAULT" で学習済みを読む
count = lambda ps: sum(p.numel() for p in ps)
print("全体", count(net.parameters()))

for p in net.parameters():
    p.requires_grad = False                 # 特徴抽出部を固定する
net.fc = nn.Linear(512, 10)                 # 最後の層だけ作り直す（10 クラス）
train = [p for p in net.parameters() if p.requires_grad]
print("学習する", count(train), " 全体", count(net.parameters()))

x = torch.zeros(2, 3, 224, 224)
print(tuple(net(x).shape))

**本書に載っている出力**
```
全体 11689512
学習する 5130  全体 11181642
(2, 10)
```

## リスト8.15　反転・切り抜き・正規化・リサイズ

In [ ]:
import torch
from torchvision import transforms as T

x = torch.arange(12.0).reshape(1, 3, 4) / 11    # 1 チャンネル 3x4、値は 0〜1
flip = T.RandomHorizontalFlip(p=1.0)
print(flip(x)[0, 0].tolist() == x[0, 0].flip(0).tolist())

crop = T.CenterCrop(2)
print(tuple(crop(x).shape))

# 正規化：(x - mean) / std。mean = std = 0.5 で [0, 1] → [-1, 1]
norm = T.Normalize([0.5], [0.5])
y = norm(x)
print(round(y.min().item(), 3), round(y.max().item(), 3))

# 大きさの変更：Resize は (高さ, 幅)
big = torch.zeros(3, 40, 60)
print(tuple(T.Resize((20, 20))(big).shape))

**本書に載っている出力**
```
True
(1, 2, 2)
-1.0 1.0
(3, 20, 20)
```

## リスト8.16　Grad-CAM を自分で計算し、重要度が重みと一致することを確かめる

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
conv = nn.Conv2d(1, 4, 3, padding=1)
fc = nn.Linear(4, 3)
x = torch.randn(1, 1, 6, 6)

A = torch.relu(conv(x))                      # 最後の畳み込みの出力 (1, 4, 6, 6)
A.retain_grad()
score = fc(A.mean(dim=(2, 3)))[0, 1]         # クラス 1 のスコア（GAP -> 全結合）
score.backward()

alpha = A.grad.mean(dim=(2, 3))[0]           # 勾配をチャンネルごとに平均
cam = torch.relu((alpha[:, None, None] * A[0]).sum(0))
print(tuple(cam.shape))
# GAP の後に全結合なら、alpha は重み / (高さ x 幅) に等しい
print(torch.allclose(alpha, fc.weight[1] / 36))

**本書に載っている出力**
```
(6, 6)
True
```

## リスト8.17　層ごとに shape を追う（出力の形を答える練習）

In [ ]:
import torch
from torch import nn

x = torch.zeros(8, 3, 32, 32)                       # (バッチ, チャンネル, 高さ, 幅)
steps = [
    ("Conv 3->16, k=3, p=1", nn.Conv2d(3, 16, 3, padding=1)),
    ("MaxPool 2", nn.MaxPool2d(2)),
    ("Conv 16->32, k=5, s=2", nn.Conv2d(16, 32, 5, stride=2)),
    ("AdaptiveAvgPool 1", nn.AdaptiveAvgPool2d(1)),
    ("Flatten", nn.Flatten()),
    ("Linear 32->10", nn.Linear(32, 10)),
]
for name, layer in steps:
    x = layer(x)
    print(name.ljust(24), tuple(x.shape))

**本書に載っている出力**
```
Conv 3->16, k=3, p=1     (8, 16, 32, 32)
MaxPool 2                (8, 16, 16, 16)
Conv 16->32, k=5, s=2    (8, 32, 6, 6)
AdaptiveAvgPool 1        (8, 32, 1, 1)
Flatten                  (8, 32)
Linear 32->10            (8, 10)
```

## リスト8.18　仮想環境の作り方（本書のコードを確認した版を指定）

（本書では未実行のリストです。環境によっては追加のインストールや通信が必要です。）

```
python -m venv .venv                       # Python 3.13.17 で確認
source .venv/bin/activate                  # Windows は .venv\Scripts\activate
pip install numpy==2.5.3 pandas==3.0.5 scikit-learn==1.9.1 matplotlib==3.11.2 seaborn==0.13.2   # 第1〜4章
pip install opencv-python==5.0.0.93 scikit-image==0.26.0 nltk==3.10.4 transformers==5.20.2   # 第6・7章
pip install fastapi==0.161.1 onnx==1.23.2 onnxruntime==1.29.0 httpx==0.28.1   # 第8章
pip install torch==2.14.1                    # 環境によって入手方法が異なる（公式の案内を確認）
pip install tensorflow-cpu==2.21.0 keras==3.15.1   # TensorFlow は第4・5章で使用
python -c "import numpy, pandas, sklearn, torch, cv2, transformers, keras; print('ok')"
```

（Python ではないので、実行せず表示だけにしています。）